<a href="https://colab.research.google.com/github/JulianBotello01/PROGCOM-B/blob/main/Copia_de_Creative_Quest_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import ipywidgets as widgets
from IPython.display import display, HTML

def flujo_tuberia(L, D, Q, rho, mu, e):
    if Q <= 0 or D <= 0:
        return 0, "Sin flujo", 0, 0, 0
    A = np.pi * (D / 2) ** 2
    v = Q / A
    Re = rho * v * D / mu

    if Re < 2300:
        tipo = "Laminar"
        f = 64 / Re
    else:
        tipo = "Turbulento"
        f = 0.25 / (np.log10(e / (3.7 * D) + 5.74 / Re**0.9))**2

    g = 9.81
    hf = f * (L / D) * (v**2 / (2 * g))
    p_in = rho * g * hf
    return Re, tipo, hf, v, p_in

def simulador(L, D, Q, rho, mu, e):
    Re, tipo, hf, v, p_in = flujo_tuberia(L, D, Q, rho, mu, e)

    fig, ax = plt.subplots(figsize=(10, 3))
    plt.close(fig)

    ax.set_facecolor("#AEE6FF")
    ax.set_xlim(-1, L + 1)
    ax.set_ylim(-1, 1)
    ax.axis("off")

    ax.fill_between([0, L], -D/2, D/2, color="#9BD3F3", alpha=0.9)
    ax.plot([0, L], [D/2, D/2], color="#E3B23C", linewidth=6)
    ax.plot([0, L], [-D/2, -D/2], color="#E3B23C", linewidth=6)

    ax.add_patch(plt.Circle((-0.3, 0), 0.15, color="#D69F0F"))
    ax.add_patch(plt.Circle((L + 0.3, 0), 0.15, color="#D69F0F"))

    n_particles = 25
    x_p = np.linspace(0, L, n_particles)
    y_p = np.random.uniform(-D/2.2, D/2.2, n_particles)
    particles, = ax.plot(x_p, y_p, "o", color="red", markersize=8, alpha=0.8)

    ax.arrow(-0.6, 0, 0.2, 0, head_width=0.05, color="darkred", linewidth=2)
    ax.text(-0.9, 0.05, "Presión Alta", color="darkred", fontsize=9)

    ax.arrow(L + 0.6, 0, -0.2, 0, head_width=0.05, color="maroon", linewidth=2)
    ax.text(L + 0.3, 0.05, "Presión Baja", color="maroon", fontsize=9)

    flow_arrow = ax.arrow(L/2 - 1, 0.7, 2, 0, head_width=0.05, color="blue", linewidth=3)
    ax.text(L/2 - 0.3, 0.8, "Flujo", color="blue", fontsize=10)

    info = ax.text(0, -0.85,
                   f"Re = {Re:.0f} | Régimen: {tipo} | hf = {hf:.3f} m",
                   fontsize=11, color="black")

    def animate(frame):
        nonlocal x_p
        if Q > 0:
            x_p = (x_p + v * 0.15) % L
        particles.set_data(x_p, y_p)
        return particles, info

    ani = animation.FuncAnimation(fig, animate, frames=200, interval=80, blit=True)
    display(HTML(ani.to_jshtml()))

L_slider = widgets.FloatSlider(value=10, min=2, max=50, step=1, description='Longitud [m]')
D_slider = widgets.FloatSlider(value=0.5, min=0.1, max=2, step=0.1, description='Diámetro [m]')
Q_slider = widgets.FloatSlider(value=0.05, min=0, max=0.2, step=0.005, description='Caudal [m³/s]')
rho_slider = widgets.FloatSlider(value=1000, min=500, max=1500, step=10, description='Densidad [kg/m³]')
mu_slider = widgets.FloatSlider(value=0.001, min=0.0001, max=0.01, step=0.0001, description='Viscosidad [Pa·s]')
e_slider = widgets.FloatSlider(value=0.0001, min=0, max=0.001, step=0.00001, description='Rugosidad [m]')

ui = widgets.VBox([
    widgets.HTML("<h3 style='color:#0d47a1'>⚙️ Control del Flujo</h3>"),
    Q_slider, D_slider, L_slider, rho_slider, mu_slider, e_slider
])

out = widgets.interactive_output(simulador, {
    'L': L_slider, 'D': D_slider, 'Q': Q_slider,
    'rho': rho_slider, 'mu': mu_slider, 'e': e_slider
})

display(widgets.HBox([ui, out]))
